# S4 · Metabolites and peripheral H


In [1]:
from pathlib import Path
import sys
base_path = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "CODE").is_dir() and (p / "DATA").is_dir()
)
sys.path.insert(0, str(Path(base_path) / 'CODE'))


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [3]:
data_types = ["microsaccades", "pupil_size"]
decimation = "decimated_2s"
field = "Hurst"
output_path = f"{base_path}/FIGURES/supplementary/Fig_S4"
os.makedirs(output_path, exist_ok=True)

# Data parameters
measures = ['EIB', 'GABA', 'Glx']

# Plotting parameters
figsize = (14, 8)
TITLE_FS = 16      
LABEL_FS = 32       
TICK_FS  = 35     
MARKERS = 100
LINE = 4
FONT_W   = 'bold'
AX_GREY  = "#424242"

vmin = 0.4
vmax = 1.8

task_colors = {
    '0Back': '#86ab00',
    '1Back': '#03b6ce',
    '2Back': '#8491ff'
}

In [4]:
from utils.model_helpers import get_task_slope_intercept

def is_hurst_field(field):
    return field in ["Hurst", "Hursts"]


point_kw = dict(s=MARKERS, alpha=0.25, linewidths=0)


def plot_measure_vs_Hurst(ax, res, df, measure, field, tasks):

    df = df.copy()
    df["Task"] = pd.Categorical(df["Task"], categories=tasks, ordered=True)
    ref_task = tasks[0]

    x_all = df[measure].to_numpy(dtype=float)
    xmin, xmax = np.nanmin(x_all), np.nanmax(x_all)
    xpad = 0.02 * (xmax - xmin if xmax > xmin else 1.0)
    xg = np.linspace(xmin, xmax, 200)

    y_min_global = np.inf
    y_max_global = -np.inf

    params = res.params
    cov = res.cov_params()
    names = params.index

    idx_intercept = names.get_loc("Intercept")
    idx_measure = names.get_loc(measure)

    for task in tasks:
        df_t = df[df["Task"] == task]

        if df_t.empty:
            continue

        x = df_t[measure].to_numpy(dtype=float)
        y = df_t[field].to_numpy(dtype=float)
        c = task_colors[task]

        ax.scatter(x, y, color=c, label=task, **point_kw)

        b0, b1 = get_task_slope_intercept(res, task, ref_task, measure)
        yg = b0 + b1 * xg

        ax.plot(xg, yg, color=c, lw=LINE)

        task_param_name = f"Task[T.{task}]"
        idx_task = names.get_loc(task_param_name) if task_param_name in names else None

        inter_names = [
            f"{measure}:Task[T.{task}]",
            f"Task[T.{task}]:{measure}",
        ]

        idx_int = None
        for nm in inter_names:
            if nm in names:
                idx_int = names.get_loc(nm)
                break

        se = np.zeros_like(xg)

        cov_array = np.asarray(cov)

        for ii, x_val in enumerate(xg):
            L = np.zeros(len(params), dtype=float)

            L[idx_intercept] = 1.0
            L[idx_measure] = x_val

            if task != ref_task and idx_task is not None:
                L[idx_task] += 1.0

            if task != ref_task and idx_int is not None:
                L[idx_int] += x_val

            var_y = float(L @ cov_array @ L)

            if var_y < 0:
                var_y = 0.0

            se[ii] = np.sqrt(var_y)

        ci_hi = yg + 1.96 * se
        ci_lo = yg - 1.96 * se

        ax.fill_between(
            xg,
            ci_lo,
            ci_hi,
            color=c,
            alpha=0.15,
            linewidth=0,
        )

        y_min_local = np.nanmin([np.nanmin(y), np.nanmin(yg), np.nanmin(ci_lo)])
        y_max_local = np.nanmax([np.nanmax(y), np.nanmax(yg), np.nanmax(ci_hi)])

        y_min_global = min(y_min_global, y_min_local)
        y_max_global = max(y_max_global, y_max_local)

    yrange = y_max_global - y_min_global if y_max_global > y_min_global else 1.0
    pad_y = 0.08 * yrange

    ylim =(0.4, 1.1) if data_type == "microsaccades" else (1, 1.7)
    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim([ylim[0], ylim[1]])

    yticks = [0.4, 0.6, 0.8, 1] if data_type == "microsaccades" else [1, 1.2, 1.4, 1.6]
    if is_hurst_field(field):
       ax.set_yticks(yticks)

    if is_hurst_field(field):
        if measure == "EIB":
            ax.set_xticks([3, 4, 5, 6])
        elif measure == "GABA":
            ax.set_xticks([1, 1.5, 2, 2.5, 3])
        elif measure == "Glx":
            ax.set_xticks([2.5, 5, 7.5, 10, 12.5, 15])

    measure_label = "Glx/GABA+" if measure == "EIB" else measure
    y_label = "H" if is_hurst_field(field) else field
    final_y_label = f"Pupil size {y_label}" if data_type == "pupil_size" else f"Microsaccades {y_label}"

    ax.set_xlabel(
        measure_label,
        fontsize=LABEL_FS,
        fontweight=FONT_W,
        color="black",
    )

    ax.set_ylabel(
        final_y_label,
        fontsize=LABEL_FS,
        fontweight=FONT_W,
        color="black",
    )

    ax.tick_params(axis="both", labelsize=TICK_FS)

    for spine in ax.spines.values():
        spine.set_color(AX_GREY)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

In [5]:
from types import SimpleNamespace

for data_type in data_types:
    lmm_results_path = f"{base_path}/RESULTS/pupillometry/{data_type}/models/MRS/{decimation}/range_1_5"

    for measure in measures:
        tasks = ["0Back", "1Back", "2Back"]
        labels = tasks

        fig, ax = plt.subplots(figsize=figsize)

        df_path = f"{lmm_results_path}/lmm_df_{measure}_R.csv"
        fixef_path = f"{lmm_results_path}/lmm_fixef_{measure}_R.csv"
        vcov_path = f"{lmm_results_path}/lmm_vcov_{measure}_R.csv"

        df = pd.read_csv(df_path)
        df["Task"] = pd.Categorical(df["Task"], categories=tasks, ordered=True)

        fixef_df = pd.read_csv(fixef_path)
        vcov_df = pd.read_csv(vcov_path)

        rename_map = {"(Intercept)": "Intercept"}

        for t in tasks:
            if t != "Rest":
                rename_map[f"Task{t}"] = f"Task[T.{t}]"
                rename_map[f"{measure}:Task{t}"] = f"{measure}:Task[T.{t}]"
                rename_map[f"Task{t}:{measure}"] = f"Task[T.{t}]:{measure}"

        fixef_df["Term"] = fixef_df["Term"].replace(rename_map)
        vcov_df["Term"] = vcov_df["Term"].replace(rename_map)
        vcov_df.columns = [rename_map.get(c, c) for c in vcov_df.columns]

        params = fixef_df.set_index("Term")["Estimate"]
        cov = vcov_df.set_index("Term")
        cov = cov.loc[params.index, params.index]

        res = SimpleNamespace(
            params=params,
            cov_params=lambda: cov
        )

        plot_measure_vs_Hurst(ax, res, df, measure, field, tasks)

        handles = [
            plt.Line2D(
                [0], [0],
                marker="o",
                markersize=12,
                linewidth=LINE,
                linestyle="-",
                color=task_colors[t],
                label=t
            )
            for t in tasks
        ]

        plt.tight_layout()

        save_path = (
            f"{output_path}/measure_vs_{field}_{measure}_"
            f"{data_type}_{decimation}_R.png"
        )

        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

        print(f"Saved: {save_path}")

Saved: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/FIGURES/supplementary/Fig_S4/measure_vs_Hurst_EIB_microsaccades_decimated_2s_R.png
Saved: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/FIGURES/supplementary/Fig_S4/measure_vs_Hurst_GABA_microsaccades_decimated_2s_R.png
Saved: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/FIGURES/supplementary/Fig_S4/measure_vs_Hurst_Glx_microsaccades_decimated_2s_R.png
Saved: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/FIGURES/supplementary/Fig_S4/measure_vs_Hurst_EIB_pupil_size_decimated_2s_R.png
Saved: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/FIGURES/supplementary/Fig_S4/measure_vs_Hurst_GABA_pupil_size_decimated_2s_R.png
Saved: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/FIGURES/supplementary/Fig_S4/measure_vs_Hurst_Glx_pupil_size_decimated_2s_R.png
